# 02 — Train GridUNet Denoiser

This notebook trains the **GridUNet denoiser** model (`barrel_denoise_grid.py`).
The denoiser operates on the 2D height field $\rho(el, az)$ grid to remove noise, repair corrupted head/pole cells, and filter bungs while preserving sharp crozehead creases.

**Sections:**
1. Setup & Device Selection
2. Model Architecture & Data Preparation
3. Training Loop with Live Loss Metrics
4. Denoised Grid Visualizations
5. Save Checkpoint

## 1. Setup & Device Selection

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.join('..', 'reconstruction'))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from barrel_denoise_grid import GridUNet, prepare_grid_inputs, DEFAULT_CHECKPOINT
from train_grid_denoiser import generate_train_sample, evaluate_model, train_epoch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using PyTorch device: {device}")
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Inspect Training Sample

Generate a synthetic sample and inspect the 4 input channels (Rho offset, Curvature, Count, Normalized polar angle).

In [ ]:
inp, target_rho, target_out, curv, el_ctr, corners = generate_train_sample(seed=42)
print(f"Input shape: {inp.shape} (4 channels x {inp.shape[1]} polar rows x {inp.shape[2]} az cols)")
print(f"Target Rho shape: {target_rho.shape}")
print(f"Target Outlier mask sum: {target_out.sum()}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
titles = ['Ch0: Rho Offset', 'Ch1: Curvature Feature', 'Ch2: Point Count Confidence', 'Target Outlier Mask']
imgs = [inp[0], inp[1], inp[2], target_out]

for ax, title, img in zip(axes.flat, titles, imgs):
    im = ax.imshow(img, aspect='auto', cmap='viridis')
    ax.set_title(title)
    ax.set_xlabel('Azimuth cell'); ax.set_ylabel('Polar row')
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 3. Training Loop

In [ ]:
epochs = 10
batch_size = 4
lr = 1e-3

model = GridUNet(in_channels=4, base_channels=32).to(device)
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
criterion_bce = nn.BCEWithLogitsLoss()

history = {'loss': [], 'rho_loss': [], 'bce_loss': [], 'val_rms': []}
best_eval_err = float('inf')

print(f"Starting training for {epochs} epochs...")
for epoch in range(1, epochs + 1):
    t0 = time.time()
    loss, rho_l, bce_l = train_epoch(model, optimizer, criterion_bce, batch_size=batch_size,
                                    samples_per_epoch=10, device=device, epoch=epoch)
    val_rms = evaluate_model(model, n_eval=3, device=device)
    t_el = time.time() - t0
    
    history['loss'].append(loss)
    history['rho_loss'].append(rho_l)
    history['bce_loss'].append(bce_l)
    history['val_rms'].append(val_rms)
    
    marker = ''
    if val_rms < best_eval_err:
        best_eval_err = val_rms
        os.makedirs(os.path.dirname(DEFAULT_CHECKPOINT), exist_ok=True)
        torch.save(model.state_dict(), DEFAULT_CHECKPOINT)
        marker = ' [saved]'
        
    print(f"Epoch {epoch:2d}/{epochs} | Loss: {loss:.4f} | Rho L1: {rho_l:.4f} | BCE: {bce_l:.4f} | Val GT RMS: {val_rms:.3f}mm | {t_el:.1f}s{marker}")

## 4. Training History Curves

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.plot(range(1, epochs + 1), history['loss'], 'b-o', label='Total Loss')
ax1.plot(range(1, epochs + 1), history['rho_loss'], 'g--', label='Rho L1 Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training Loss', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(range(1, epochs + 1), history['val_rms'], 'r-s', label='Val GT RMS (mm)')
ax2.set_ylabel('Val GT RMS (mm)', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title('GridUNet Training & Validation History')
plt.grid(True)
plt.show()